In [2]:
from dotenv import load_dotenv
import os
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

print(NEO4J_URI)
print(NEO4J_USERNAME)

neo4j+s://8232f16f.databases.neo4j.io
neo4j


In [3]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Neo4j connection successful!")

Neo4j connection successful!


In [4]:
import pandas as pd

df_relationships = pd.read_csv(
    "../data/product_relationships.csv"
)

print(df_relationships.shape)
df_relationships.head()

(265, 7)


,Product_A,Product_A_Name,Product_B,Product_B_Name,Purchase_Count,Confidence,Lift
0,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,428,0.773960,19.218577
1,22386,JUMBO BAG PINK POLKADOT,85099B,JUMBO BAG RED RETROSPOT,415,0.600579,7.368768
2,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,410,0.663430,15.039361
3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,368,0.665461,23.676167
4,23203,JUMBO BAG DOILEY PATTERNS,85099B,JUMBO BAG RED RETROSPOT,356,0.416862,5.114662


In [5]:
df_relationships.info()

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Product_A       265 non-null    str    
 1   Product_A_Name  265 non-null    str    
 2   Product_B       265 non-null    str    
 3   Product_B_Name  265 non-null    str    
 4   Purchase_Count  265 non-null    int64  
 5   Confidence      265 non-null    float64
 6   Lift            265 non-null    float64
dtypes: float64(2), int64(1), str(4)
memory usage: 14.6 KB


In [6]:
df_relationships[
    [
        "Product_A",
        "Product_A_Name",
        "Product_B",
        "Product_B_Name",
        "Purchase_Count",
        "Confidence",
        "Lift"
    ]
].head(10)

,Product_A,Product_A_Name,Product_B,Product_B_Name,Purchase_Count,Confidence,Lift
0,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,428,0.773960,19.218577
1,22386,JUMBO BAG PINK POLKADOT,85099B,JUMBO BAG RED RETROSPOT,415,0.600579,7.368768
2,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,410,0.663430,15.039361
3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,368,0.665461,23.676167
4,23203,JUMBO BAG DOILEY PATTERNS,85099B,JUMBO BAG RED RETROSPOT,356,0.416862,5.114662
5,20725,LUNCH BAG RED RETROSPOT,22384,LUNCH BAG PINK POLKADOT,352,0.368201,8.648013
6,82482,WOODEN PICTURE FRAME WHITE FINISH,82494L,WOODEN FRAME ANTIQUE WHITE,350,0.527108,12.863917
7,20725,LUNCH BAG RED RETROSPOT,22383,LUNCH BAG SUKI DESIGN,346,0.361925,7.618466
8,20725,LUNCH BAG RED RETROSPOT,20727,LUNCH BAG BLACK SKULL.,346,0.361925,7.597986
9,22698,PINK REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,342,0.779043,19.344796


In [7]:
df_relationships.shape

(265, 7)

In [8]:
from neo4j import GraphDatabase

def create_product_relationships(tx, row):
    query = """
    MERGE (a:Product {stock_code: $product_a})
    SET a.name = $product_a_name

    MERGE (b:Product {stock_code: $product_b})
    SET b.name = $product_b_name

    MERGE (a)-[r:BOUGHT_WITH]->(b)
    SET r.purchase_count = $purchase_count,
        r.confidence = $confidence,
        r.lift = $lift
    """

    tx.run(
        query,
        product_a=str(row["Product_A"]),
        product_a_name=row["Product_A_Name"],
        product_b=str(row["Product_B"]),
        product_b_name=row["Product_B_Name"],
        purchase_count=int(row["Purchase_Count"]),
        confidence=float(row["Confidence"]),
        lift=float(row["Lift"])
    )


with driver.session() as session:
    for _, row in df_relationships.iterrows():
        session.execute_write(create_product_relationships, row)

print("Products and relationships added successfully!")

Products and relationships added successfully!


In [9]:
import sys

sys.path.append("../")

from src.neo4j_client import Neo4jClient

In [10]:
client = Neo4jClient()

client.verify_connection()

print("Neo4j client working!")

Neo4j client working!


In [11]:
recommendations = client.get_recommendations("22697", limit=5)

recommendations

[{'stock_code': '22698',
  'name': 'PINK REGENCY TEACUP AND SAUCER',
  'purchase_count': 368,
  'confidence': 0.6654611211573237,
  'lift': 23.67616685958141},
 {'stock_code': '22699',
  'name': 'ROSES REGENCY TEACUP AND SAUCER ',
  'purchase_count': 428,
  'confidence': 0.7739602169981917,
  'lift': 19.21857651716177}]

In [12]:
from src.recommendation import RecommendationEngine

engine = RecommendationEngine()

recommendations = engine.recommend("22697", limit=5)

recommendations

[{'stock_code': '22698',
  'name': 'PINK REGENCY TEACUP AND SAUCER',
  'purchase_count': 368,
  'confidence': 0.6654611211573237,
  'lift': 23.67616685958141},
 {'stock_code': '22699',
  'name': 'ROSES REGENCY TEACUP AND SAUCER ',
  'purchase_count': 428,
  'confidence': 0.7739602169981917,
  'lift': 19.21857651716177}]

In [13]:
from src.data_processing import (
    load_transactions,
    clean_transactions,
    load_product_relationships
)

In [14]:
df_relationships = load_product_relationships(
    "../data/product_relationships.csv"
)

df_relationships.shape

(265, 7)

In [15]:
df_relationships.head()

,Product_A,Product_A_Name,Product_B,Product_B_Name,Purchase_Count,Confidence,Lift
0,22697,GREEN REGENCY TEACUP AND SAUCER,22699,ROSES REGENCY TEACUP AND SAUCER,428,0.773960,19.218577
1,22386,JUMBO BAG PINK POLKADOT,85099B,JUMBO BAG RED RETROSPOT,415,0.600579,7.368768
2,22726,ALARM CLOCK BAKELIKE GREEN,22727,ALARM CLOCK BAKELIKE RED,410,0.663430,15.039361
3,22697,GREEN REGENCY TEACUP AND SAUCER,22698,PINK REGENCY TEACUP AND SAUCER,368,0.665461,23.676167
4,23203,JUMBO BAG DOILEY PATTERNS,85099B,JUMBO BAG RED RETROSPOT,356,0.416862,5.114662


In [2]:
import sys
import os

sys.path.append(
    os.path.abspath("../")
)

from src.rag import ProductRAG

In [3]:
print("RAG imported successfully!")

RAG imported successfully!


In [4]:
rag = ProductRAG(
    product_file="../data/products.csv",
    persist_directory="../data/chroma_db"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

d:\razorpay-ai-commerce-agent\src\rag.py:64: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vectorstore = Chroma(


Loaded 3958 products into ChromaDB.


In [5]:
results = rag.search(
    "green regency teacup",
    k=5
)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------
Product Name: REGENCY TEA PLATE GREEN 
Stock Code: 23171
--------------------------------------------------
Product Name: PINK REGENCY TEACUP AND SAUCER
Stock Code: 22698
--------------------------------------------------
Product Name: ROSES REGENCY TEACUP AND SAUCER 
Stock Code: 22699
--------------------------------------------------
Product Name: REGENCY TEA SPOON
Stock Code: 23160
--------------------------------------------------


In [6]:
from src.agent import CommerceAgent

In [7]:
agent = CommerceAgent()

print("Agent initialized successfully!")

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/adapter_config.json


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_co

KeyboardInterrupt: 

In [8]:
recommendations = agent.recommend_product(
    "22697",
    limit=5
)

recommendations

ServiceUnavailable: Failed to DNS resolve address 8232f16f.databases.neo4j.io:7687: [Errno 11001] getaddrinfo failed

In [1]:
results = agent.search_product(
    "green regency teacup",
    k=5
)

for doc in results:
    print(doc.page_content)
    print("-" * 50)

NameError: name 'agent' is not defined

In [ ]:
recommendations = agent.recommend_product(
    "22697",
    limit=5
)

recommendations

[{'stock_code': '22698',
  'name': 'PINK REGENCY TEACUP AND SAUCER',
  'purchase_count': 368,
  'confidence': 0.6654611211573237,
  'lift': 23.67616685958141},
 {'stock_code': '22699',
  'name': 'ROSES REGENCY TEACUP AND SAUCER ',
  'purchase_count': 428,
  'confidence': 0.7739602169981917,
  'lift': 19.21857651716177}]

In [ ]:
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

groq/compound
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-20b
qwen/qwen3.6-27b
canopylabs/orpheus-arabic-saudi
groq/compound-mini
openai/gpt-oss-safeguard-20b
whisper-large-v3
allam-2-7b
openai/gpt-oss-120b
canopylabs/orpheus-v1-english
qwen/qwen3.8-27b


In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

response = llm.invoke(
    "What is an AI commerce recommendation system?"
)

print(response.content)

## What Is an AI Commerce Recommendation System?

An **AI commerce recommendation system** (often just called a *recommendation engine*) is a machine‑learning‑driven component of an online retail or digital‑content platform that predicts and presents items a user is most likely to buy, click, or engage with.  
It sits between the user’s browsing experience and the product catalog, automatically filtering and ranking millions of items so that each visitor sees a personalized, relevant set of suggestions.

---

### Core Functions

| Function | What It Does | Typical AI Technique |
|----------|--------------|----------------------|
| **Item Discovery** | Finds new or hidden products that match a user’s interests. | Collaborative filtering, content‑based filtering, hybrid models |
| **Personalization** | Tailors the list to each user’s past behavior, demographics, or context. | User embeddings, neural collaborative filtering, transformers |
| **Ranking & Scoring** | Orders items by predict

In [2]:
import sys
import os

sys.path.append(os.path.abspath("../"))

print(os.path.abspath("../"))
from src.agent import CommerceAgent

d:\razorpay-ai-commerce-agent


In [3]:
agent = CommerceAgent()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

d:\razorpay-ai-commerce-agent\src\rag.py:64: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vectorstore = Chroma(


Loaded 3958 products into ChromaDB.


In [4]:
answer = agent.ask_llm(
    "Explain what product recommendations are in one sentence."
)

print(answer)

Product recommendations are personalized suggestions of items to customers, generated by analyzing their browsing or purchase history to increase relevance and drive sales.


In [5]:
agent.recommend_product("22697", limit=5)

[{'stock_code': '22698',
  'name': 'PINK REGENCY TEACUP AND SAUCER',
  'purchase_count': 368,
  'confidence': 0.6654611211573237,
  'lift': 23.67616685958141},
 {'stock_code': '22699',
  'name': 'ROSES REGENCY TEACUP AND SAUCER ',
  'purchase_count': 428,
  'confidence': 0.7739602169981917,
  'lift': 19.21857651716177}]

In [6]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print(os.getenv("NEO4J_URI"))

neo4j+s://8232f16f.databases.neo4j.io


In [7]:
from src.neo4j_client import Neo4jClient

neo4j = Neo4jClient()

print(neo4j.get_recommendations("22697", limit=5))

[{'stock_code': '22698', 'name': 'PINK REGENCY TEACUP AND SAUCER', 'purchase_count': 368, 'confidence': 0.6654611211573237, 'lift': 23.67616685958141}, {'stock_code': '22699', 'name': 'ROSES REGENCY TEACUP AND SAUCER ', 'purchase_count': 428, 'confidence': 0.7739602169981917, 'lift': 19.21857651716177}]


In [8]:
recommendations = agent.recommend_product(
    "22697",
    limit=5
)

print(recommendations)

[{'stock_code': '22698', 'name': 'PINK REGENCY TEACUP AND SAUCER', 'purchase_count': 368, 'confidence': 0.6654611211573237, 'lift': 23.67616685958141}, {'stock_code': '22699', 'name': 'ROSES REGENCY TEACUP AND SAUCER ', 'purchase_count': 428, 'confidence': 0.7739602169981917, 'lift': 19.21857651716177}]


In [9]:
print(agent.ask_llm(
    "A customer bought product 22697. "
    "Explain why a recommendation system might suggest other products."
))

When a customer checks out product **22697**, a recommendation engine can surface other items for a variety of reasons.  Below is a quick‑look at the most common motivations and the data sources that feed them.

| Why the system suggests something else | What data or logic is used | Typical business goal |
|----------------------------------------|---------------------------|-----------------------|
| **Collaborative filtering** – “people who bought 22697 also bought …” | Purchase histories of many users; similarity scores between customers or items | Cross‑sell, increase basket size |
| **Content‑based filtering** – “this item shares attributes with 22697” | Product metadata (category, brand, specs, tags) | Keep the recommendation relevant to the user’s interests |
| **Association rules / Market‑basket analysis** – “22697 + X → Y” | Frequent itemsets mined from transaction logs | Bundle or upsell complementary products |
| **Hybrid models** – combine the above with contextual signals 

In [1]:
import sys
import os

sys.path.append(os.path.abspath("../"))

In [2]:
from src.agent import ask_agent

d:\razorpay-ai-commerce-agent\src\rag.py:44: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
response = ask_agent(
    "I bought product 22697. What products should I buy with it?"
)

print(response)

Here are the top products that customers who bought **22697** also liked:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| **22698** | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| **22699** | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

- **Confidence** shows how often the product is bought together with 22697.  
- **Lift** indicates how much more likely the pair is bought together compared to random chance.

If you’d like more suggestions or want to see products in a specific category, just let me know!


In [1]:
import sys
import os

sys.path.append(os.path.abspath("../"))

In [2]:
from src.agent import ask_agent

d:\razorpay-ai-commerce-agent\src\rag.py:44: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
response = ask_agent(
    "I bought product 22697. What products should I buy with it?"
)

print(response)

Here are the two products that customers most often buy together with **Product 22697**:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| **22698** | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| **22699** | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

**Why these?**  
- Both are complementary tea‑service items that pair nicely with the item you already own.  
- The high lift values (23.68 and 19.22) indicate a strong association—customers who buy 22697 are much more likely to add these to their cart than random shoppers.  

If you’re looking to complete a set or add a matching accessory, these are the top picks. Let me know if you’d like more options or details on any of them!


In [5]:
response = ask_agent(
    "Find products similar to GREEN REGENCY TEACUP AND SAUCER"
)

print(response)

Here are the products that are most frequently bought together with the **GREEN REGENCY TEACUP AND SAUCER (Stock Code 22697)**:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| **22698** | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| **22699** | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

**Why these are similar**

- Both are part of the same “Reggie” teacup series, so they share the same design, size, and material.
- The high lift values (23.68 and 19.22) indicate that customers who buy the green set are much more likely to also buy the pink or roses set than would be expected by chance.
- The confidence scores (0.665 and 0.774) show that a substantial proportion of green‑set purchasers also buy these alternatives.

**Next steps**

- If you’re looking to upsell or bundle, consider offering a discount on the pink or roses set when a customer adds the green set to their cart.

In [6]:
from src.agent import search_products

result = search_products.invoke({
    "query": "GREEN REGENCY TEACUP AND SAUCER",
    "k": 5
})

print(result)

['Product Name: GREEN REGENCY TEACUP AND SAUCER\nStock Code: 22697', 'Product Name: GREEN REGENCY TEACUP AND SAUCER\nStock Code: 22697', 'Product Name: GREEN REGENCY TEACUP AND SAUCER\nStock Code: 22697', 'Product Name: GREEN REGENCY TEACUP AND SAUCER\nStock Code: 22697', 'Product Name: PINK REGENCY TEACUP AND SAUCER\nStock Code: 22698']


In [1]:
import sys
import os

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

d:\razorpay-ai-commerce-agent


In [2]:
from src.rag import ProductRAG

rag = ProductRAG(
    product_file="../data/products.csv",
    persist_directory="../data/chroma_db"
)

Loaded 3958 unique products


d:\razorpay-ai-commerce-agent\src\rag.py:61: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


KeyboardInterrupt: 

In [4]:
result = rag.search(
    "GREEN REGENCY TEACUP AND SAUCER",
    k=5
)

for item in result:
    print(item.page_content)
    print("-" * 50)

Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------
Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------
Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------
Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------
Product Name: GREEN REGENCY TEACUP AND SAUCER
Stock Code: 22697
--------------------------------------------------


In [6]:
print(len(rag.documents))

3958


In [7]:
for doc in rag.documents[:10]:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

Product Name: WHITE HANGING HEART T-LIGHT HOLDER
Stock Code: 85123A
{'stock_code': '85123A', 'name': 'WHITE HANGING HEART T-LIGHT HOLDER'}
--------------------------------------------------
Product Name: WHITE METAL LANTERN
Stock Code: 71053
{'stock_code': '71053', 'name': 'WHITE METAL LANTERN'}
--------------------------------------------------
Product Name: CREAM CUPID HEARTS COAT HANGER
Stock Code: 84406B
{'stock_code': '84406B', 'name': 'CREAM CUPID HEARTS COAT HANGER'}
--------------------------------------------------
Product Name: KNITTED UNION FLAG HOT WATER BOTTLE
Stock Code: 84029G
{'stock_code': '84029G', 'name': 'KNITTED UNION FLAG HOT WATER BOTTLE'}
--------------------------------------------------
Product Name: RED WOOLLY HOTTIE WHITE HEART.
Stock Code: 84029E
{'stock_code': '84029E', 'name': 'RED WOOLLY HOTTIE WHITE HEART.'}
--------------------------------------------------
Product Name: SET 7 BABUSHKA NESTING BOXES
Stock Code: 22752
{'stock_code': '22752', 'name': 'SE

In [2]:
import sys
import os

sys.path.append(os.path.abspath("../"))
from src.agent import ask_agent

Loaded 3958 unique products


d:\razorpay-ai-commerce-agent\src\rag.py:61: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB loaded successfully


In [3]:
print(
    ask_agent(
        "Find products similar to product 22697",
        thread_id="similar_test"
    )
)

Here are products that are semantically similar to **Product 22697**:

| Stock Code | Product Name |
|------------|--------------|
| 22698 | PINK REGENCY TEACUP AND SAUCER |
| 22699 | ROSES REGENCY TEACUP AND SAUCER |
| 23171 | REGENCY TEA PLATE GREEN |
| 23160 | REGENCY TEA SPOON |
| 22072 | RED RETROSPOT TEA CUP AND SAUCER |

These items were retrieved via semantic similarity search and share contextual relevance with the original product.


In [2]:
print(
    ask_agent(
        "I bought product 22697. What products should I buy with it?",
        thread_id="test_1"
    )
)

Here are the top products most frequently bought together with **22697**:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

- **Purchase Count** shows how many times the pair was bought together.  
- **Confidence** (0–1) indicates the likelihood that a customer who buys 22697 will also buy the recommended item.  
- **Lift** measures how much more often the pair occurs together than expected by chance; higher values mean a stronger association.  

These two teacup sets are the most common companions for product 22697.


In [3]:
print(
    ask_agent(
        "Find products similar to GREEN REGENCY TEACUP AND SAUCER",
        thread_id="test_2"
    )
)

Here are products that are semantically similar to **GREEN REGENCY TEACUP AND SAUCER**:

| Product Name | Stock Code |
|--------------|------------|
| GREEN REGENCY TEACUP AND SAUCER | 22697 |
| PINK REGENCY TEACUP AND SAUCER | 22698 |
| ROSES REGENCY TEACUP AND SAUCER | 22699 |
| REGENCY TEA PLATE GREEN | 23171 |
| REGENCY TEA SPOON | 23160 |

These items were retrieved based on semantic similarity to your query.


In [4]:
print(
    ask_agent(
        "What should I buy with product 999999?",
        thread_id="test_3"
    )
)

I’m sorry, but I couldn’t find any products that are frequently bought together with product 999999. If you’d like, I can help you search for similar items or explore other options.


In [4]:
print(
    ask_agent(
        "Find products similar to GREEN REGENCY TEACUP AND SAUCER",
        thread_id="customer_1"
    )
)

Here are the products that the semantic search returned as most similar to **GREEN REGENCY TEACUP AND SAUCER**:

| Product Name | Stock Code |
|--------------|------------|
| GREEN REGENCY TEACUP AND SAUCER | 22697 |
| PINK REGENCY TEACUP AND SAUCER | 22698 |
| ROSES REGENCY TEACUP AND SAUCER | 22699 |
| REGENCY TEA PLATE GREEN | 23171 |
| REGENCY TEA SPOON | 23160 |

These items share similar naming patterns and likely belong to the same product family or collection.


In [2]:
print(
    ask_agent(
        "Which one is pink?",
        thread_id="customer_1"
    )
)

I’m not sure which items you’re referring to. Could you let me know the product names or the list you’re looking at? That way I can tell you which one is pink.


In [3]:
print(
    ask_agent(
        "Which product did we discuss?",
        thread_id="customer_2"
    )
)

I’m not sure which product you’re referring to. Could you let me know the product name or any details you have in mind?


In [3]:
response = ask_agent(
    "I bought product 22697. What products should I buy with it?"
)

print(response)

Here are the top products most frequently bought together with **22697**:

| Stock Code | Product | Purchase Count | Confidence | Lift |
|------------|---------|----------------|------------|------|
| **22698** | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| **22699** | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

- **Purchase Count** – how many times the pair was bought together.  
- **Confidence** – probability that the second item is bought when the first is purchased.  
- **Lift** – how much more likely the pair is bought together compared to random chance (higher is stronger association).

These two teacup sets are the most common companions for product 22697.


In [4]:
response = ask_agent(
    "Find products similar to GREEN REGENCY TEACUP AND SAUCER"
)

print(response)

**Products similar to “GREEN REGENCY TEACUP AND SAUCER”**

| Product Name | Stock Code |
|--------------|------------|
| GREEN REGENCY TEACUP AND SAUCER | 22697 |
| PINK REGENCY TEACUP AND SAUCER | 22698 |
| ROSES REGENCY TEACUP AND SAUCER | 22699 |
| REGENCY TEA PLATE GREEN | 23171 |
| REGENCY TEA SPOON | 23160 |

These items were retrieved via semantic similarity search and match the style, brand, or theme of the green Regency set.


In [5]:
print(
    ask_agent(
        "My product is 22697. What products should I buy with it?",
        thread_id="customer_2"
    )
)

Here are the top products that customers often buy together with **22697**:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

These items have the highest purchase counts and strong association metrics (confidence and lift) with product 22697.


In [6]:
print(
    ask_agent(
        "Which one is pink?",
        thread_id="customer_2"
    )
)

The pink one is **stock code 22698 – “PINK REGENCY TEACUP AND SAUCER.”**


In [5]:
print(
    ask_agent(
        "I am interested in product 22697",
        thread_id="customer_2"
    )
)

print(
    ask_agent(
        "Which product am I interested in?",
        thread_id="customer_2"
    )
)

I couldn’t find a direct record for stock code **22697** in the catalog, but here are a few items that are semantically similar based on the search:

| Product Name | Stock Code |
|--------------|------------|
| PINK REGENCY TEACUP AND SAUCER | 22698 |
| ROSES REGENCY TEACUP AND SAUCER | 22699 |
| REGENCY TEA PLATE GREEN | 23171 |
| REGENCY TEA SPOON | 23160 |
| RED RETROSPOT TEA CUP AND SAUCER | 22072 |

These items share similar themes or styles (e.g., Regency‑style tea sets, teacups, and saucers). If you’d like to see products that are frequently bought together with any of these, just let me know!
You mentioned that you’re interested in **product 22697**.


In [7]:
from src.agent import search_products

In [8]:
print(search_products.invoke({
    "query": "22697",
    "k": 5
}))

['Product Name: PINK REGENCY TEACUP AND SAUCER\nStock Code: 22698', 'Product Name: ROSES REGENCY TEACUP AND SAUCER \nStock Code: 22699', 'Product Name: REGENCY TEA PLATE GREEN \nStock Code: 23171', 'Product Name: REGENCY TEA SPOON\nStock Code: 23160', 'Product Name: RED RETROSPOT TEA CUP AND SAUCER \nStock Code: 22072']


In [9]:
print(
    ask_agent(
        "Find products similar to product 22697",
        thread_id="customer_4"
    )
)

Here are five products that are semantically similar to **Product 22697** (PINK REGENCY TEACUP AND SAUCER):

| Stock Code | Product Name |
|------------|--------------|
| 22698 | PINK REGENCY TEACUP AND SAUCER |
| 22699 | ROSES REGENCY TEACUP AND SAUCER |
| 23171 | REGENCY TEA PLATE GREEN |
| 23160 | REGENCY TEA SPOON |
| 22072 | RED RETROSPOT TEA CUP AND SAUCER |

These items share similar themes or styles (e.g., Regency‑style tea sets, teacups, saucers, and related accessories).


In [10]:
print(
    ask_agent(
        "What products are frequently bought together with product 22697?",
        thread_id="customer_5"
    )
)

**Products frequently bought together with 22697**

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

These items have the highest co‑purchase frequency with product 22697.


In [11]:
print(
    ask_agent(
        "Give me the best recommendations for product 22697",
        thread_id="customer_6"
    )
)

**Products frequently bought together with 22697**

| Stock Code | Name | Purchase Count | Confidence | Lift |
|------------|------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

These items have the highest purchase counts and strong association metrics (confidence and lift) with product 22697, making them the top recommendations for cross‑selling or bundling.


In [12]:
print(
    ask_agent(
        "What products are similar to product 22697?",
        thread_id="test_similarity_1"
    )
)

Here are five products that are semantically similar to product **22697**:

| Stock Code | Product Name |
|------------|--------------|
| 22698 | PINK REGENCY TEACUP AND SAUCER |
| 22699 | ROSES REGENCY TEACUP AND SAUCER |
| 23171 | REGENCY TEA PLATE GREEN |
| 23160 | REGENCY TEA SPOON |
| 22072 | RED RETROSPOT TEA CUP AND SAUCER |

These items were retrieved via a semantic search and share similar themes or styles with product 22697.


In [13]:
print(
    ask_agent(
        "What products are frequently bought together with 22697?",
        thread_id="test_purchase_1"
    )
)

**Products frequently bought together with 22697**

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

These items are the top two that customers most often purchase alongside product 22697.


In [14]:
print(
    ask_agent(
        "I am interested in product 22697.",
        thread_id="customer_10"
    )
)

print(
    ask_agent(
        "Which product am I interested in?",
        thread_id="customer_10"
    )
)

Here are the products most frequently bought together with **22697**:

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

These items have the highest purchase counts, confidence, and lift values when paired with product 22697.
You’re interested in product **22697**.


In [15]:
ask_agent("My product is 22697", thread_id="customer_2")

ask_agent("What did we discuss?", thread_id="customer_2")

'**Summary of our conversation**\n\n1. **User’s interest** – You mentioned you’re looking for product **22697**.  \n2. **Search attempt** – I performed a semantic search for that stock code.  \n3. **Result** – No exact match for 22697 was found. The search returned five semantically similar items (Regency‑style teacups and saucers).  \n4. **Next steps** – I offered to provide cross‑sell or bundle suggestions for the similar items if that would help.'

In [16]:
ask_agent("What did we discuss?", thread_id="customer_3")

'We’ve been chatting about how to use the two tools available for product recommendations and searches. I explained when to use each tool, how to interpret their outputs, and the rules for describing results. No actual tool calls were made yet.'

In [17]:
print(
    ask_agent(
        "What products are frequently bought together with product 22697?",
        thread_id="test_1"
    )
)

**Products frequently bought together with 22697**

| Stock Code | Product Name | Purchase Count | Confidence | Lift |
|------------|--------------|----------------|------------|------|
| 22698 | PINK REGENCY TEACUP AND SAUCER | 368 | 0.665 | 23.68 |
| 22699 | ROSES REGENCY TEACUP AND SAUCER | 428 | 0.774 | 19.22 |

These items have the highest co‑purchase frequency with product 22697.


In [18]:
print(
    ask_agent(
        "Find products similar to product 22697.",
        thread_id="test_2"
    )
)

Here are products that are semantically similar to **product 22697**:

| Product Name | Stock Code |
|--------------|------------|
| PINK REGENCY TEACUP AND SAUCER | 22698 |
| ROSES REGENCY TEACUP AND SAUCER | 22699 |
| REGENCY TEA PLATE GREEN | 23171 |
| REGENCY TEA SPOON | 23160 |
| RED RETROSPOT TEA CUP AND SAUCER | 22072 |

These items were retrieved via semantic similarity search and share contextual relevance with product 22697.


In [19]:
print(
    ask_agent(
        "Give me the best recommendations for product 22697 using both purchase history and similarity.",
        thread_id="test_3"
    )
)

**Recommendations for product 22697**

| Stock Code | Product Name | Source | Notes |
|------------|--------------|--------|-------|
| **22698** | PINK REGENCY TEACUP AND SAUCER | Purchase history | Bought together 368 times with 22697 (confidence ≈ 0.67, lift ≈ 23.68) |
| **22699** | ROSES REGENCY TEACUP AND SAUCER | Purchase history | Bought together 428 times with 22697 (confidence ≈ 0.77, lift ≈ 19.22) |
| **23171** | REGENCY TEA PLATE GREEN | Similarity | Semantically similar to 22697 |
| **23160** | REGENCY TEA SPOON | Similarity | Semantically similar to 22697 |
| **22072** | RED RETROSPOT TEA CUP AND SAUCER | Similarity | Semantically similar to 22697 |

These five items combine the strongest purchase‑history links (22698, 22699) with the most semantically similar products from the catalog (23171, 23160, 22072).


In [20]:
print(
    ask_agent(
        "I am interested in product 22697.",
        thread_id="customer_10"
    )
)

print(
    ask_agent(
        "What product am I interested in?",
        thread_id="customer_10"
    )
)

I couldn’t locate product **22697** in the catalog.  
Here are the products that came up as semantically similar when searching for that stock code:

- **PINK REGENCY TEACUP AND SAUCER** – Stock Code 22698  
- **ROSES REGENCY TEACUP AND SAUCER** – Stock Code 22699  
- **REGENCY TEA PLATE GREEN** – Stock Code 23171  
- **REGENCY TEA SPOON** – Stock Code 23160  
- **RED RETROSPOT TEA CUP AND SAUCER** – Stock Code 22072  

If you have a different product name or additional details, let me know and I can refine the search.
You’re interested in product **22697**.
